# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Task Type: Ranking

My lane is a ranking problem because the goal is to prioritize content pages for refresh. Instead of simply predicting whether a page should be refreshed, the model ranks pages so that the highest-priority pages appear first. This helps content teams decide where to focus their limited time and resources.


In [2]:

print("Task Type: Ranking")



Task Type: Ranking


In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 2. Target or proxy

The target is the refresh priority of each content page.

The dataset does not contain a direct "refresh priority" label, so I will use the `trend_direction` column as a proxy. Pages with a downward trend are more likely to need a content refresh.

This target comes from an observed outcome in the historical data rather than a manually defined rule.


In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["trend_direction"].value_counts()



,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


## 3. Success metric

Success Metric: NDCG@10 (Normalized Discounted Cumulative Gain)

NDCG@10 measures how well the model ranks the highest-priority pages near the top of the list. A higher score means the model is better at recommending which pages should be refreshed first.


In [5]:
df["trend_direction"].value_counts(normalize=True)


,proportion
trend_direction,
down,0.542067
stable,0.198733
up,0.146267
new,0.074533
flat,0.038400


## 4. The unit of analysis

One row represents one content page.

Each row contains historical information about a single page, including its content type, search intent, trend direction, and other measured features that may help determine refresh priority.


In [6]:
 print("Dataset Shape:", df.shape)

df.head()


Dataset Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
df[
    [
        "content_type",
        "main_intent",
        "trend_direction"
    ]
].head()


,content_type,main_intent,trend_direction
0,keyword article,transactional,down
1,keyword article,informational,down
2,keyword article,informational,down
3,keyword article,commercial,stable
4,keyword article,informational,down


In [8]:
df["refresh_priority"] = (df["trend_direction"] == "down").astype(int)

df[["trend_direction", "refresh_priority"]].head()


,trend_direction,refresh_priority
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1


## 5. Why ML beats a fixed rule here

A fixed rule such as "refresh every page with a downward trend" ignores other useful information, such as content type, search intent, and historical performance.

Machine learning can learn patterns across many measured features at the same time and produce a ranked list of pages based on their overall likelihood of benefiting from a refresh.

The model provides decision-support for content teams, while the final refresh decisions remain with people.


In [9]:
# Show the features available for learning patterns
print("Number of features:", len(df.columns))

df.columns.tolist()



Number of features: 45


['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct',
 'refresh_priority']

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.